In [0]:
from pyspark.sql.functions import (
    col, round, max, min, avg, count,
    last, first, current_timestamp, lit,
    dense_rank, sum, when
)
from pyspark.sql.window import Window

# Storage config
storage_account = "theportfoliostorage"
container = "datalake"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", 
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", 
               dbutils.secrets.get(scope="de-portfolio-scope", key="sp-client-id"))
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", 
               dbutils.secrets.get(scope="de-portfolio-scope", key="sp-client-secret"))
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", 
               f"https://login.microsoftonline.com/{dbutils.secrets.get(scope='de-portfolio-scope', key='sp-tenant-id')}/oauth2/token")

# Read from Silver
df_silver = spark.table("crypto.silver.prices_clean")

# Work with non-null previous prices only for movement calculations
df_reportable = df_silver.filter(col("previous_price").isNotNull())

print(f"Total Silver rows: {df_silver.count()}")
print(f"Reportable rows (excluding first batch): {df_reportable.count()}")

In [0]:
# Gold Model 1: Latest price snapshot per coin
# Most recent batch only - what Power BI shows as current state

latest_batch = df_silver.agg(max("_batch_id")).collect()[0][0]

df_latest_prices = df_silver \
    .filter(col("_batch_id") == latest_batch) \
    .select(
        "symbol", "name", "current_price",
        "market_cap", "market_cap_rank",
        "total_volume", "high_24h", "low_24h",
        "price_change_24h", "price_change_percentage_24h",
        "circulating_supply", "_ingested_at"
    ) \
    .withColumn("_gold_processed_at", current_timestamp())

print(f"Latest prices: {df_latest_prices.count()} coins")
df_latest_prices.select(
    "name", "current_price", 
    "price_change_percentage_24h", "market_cap_rank"
).orderBy("market_cap_rank").show(truncate=False)

In [0]:
# Gold Model 2: Price movement summary per coin
# High, low, range, average, direction counts

volatility_window = Window.orderBy(col("price_range").desc())

df_price_summary = df_silver \
    .groupBy("symbol", "name") \
    .agg(
        round(min("current_price"), 2).alias("min_price"),
        round(max("current_price"), 2).alias("max_price"),
        round(avg("current_price"), 2).alias("avg_price"),
        round(max("current_price") - min("current_price"), 2).alias("price_range"),
        round(avg("batch_price_change_pct"), 4).alias("avg_batch_change_pct"),
        count(when(col("price_direction") == "UP", 1)).alias("up_count"),
        count(when(col("price_direction") == "DOWN", 1)).alias("down_count"),
        count(when(col("price_direction") == "FLAT", 1)).alias("flat_count"),
        count(when(col("is_volatile") == True, 1)).alias("volatile_batches")
    ) \
    .withColumn("volatility_rank", dense_rank().over(volatility_window)) \
    .withColumn("overall_direction",
        when(col("up_count") > col("down_count"), lit("BULLISH"))
        .when(col("down_count") > col("up_count"), lit("BEARISH"))
        .otherwise(lit("NEUTRAL"))) \
    .withColumn("_gold_processed_at", current_timestamp())

print(f"Price summary: {df_price_summary.count()} coins")
df_price_summary.orderBy("volatility_rank").show(truncate=False)

In [0]:
# Gold Model 3: Market dominance
# Each coin's share of total market cap

total_market_cap = df_latest_prices.agg(
    sum("market_cap")).collect()[0][0]

df_market_dominance = df_latest_prices \
    .withColumn("market_cap_share_pct",
        round(col("market_cap") / total_market_cap * 100, 4)) \
    .select(
        "symbol", "name", "market_cap",
        "market_cap_share_pct", "market_cap_rank",
        "current_price", "total_volume"
    ) \
    .orderBy("market_cap_rank") \
    .withColumn("_gold_processed_at", current_timestamp())

print(f"Market dominance: {df_market_dominance.count()} coins")
df_market_dominance.show(truncate=False)

In [0]:
# Write Gold Model 1: Latest prices
df_latest_prices.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("crypto.gold.latest_prices")

print("Latest prices written")

# Write Gold Model 2: Price movement summary
df_price_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("crypto.gold.price_movement_summary")

print("Price movement summary written")

# Write Gold Model 3: Market dominance
df_market_dominance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("crypto.gold.market_dominance")

print("Market dominance written")

# Final verification
print("\n=== Gold Layer Verification ===")
spark.sql("SELECT COUNT(*) as coins FROM crypto.gold.latest_prices").show()
spark.sql("SELECT COUNT(*) as coins FROM crypto.gold.price_movement_summary").show()
spark.sql("SELECT COUNT(*) as coins FROM crypto.gold.market_dominance").show()